# Ball fine-tune on free Colab (ball fix plan F4-F5)

Runtime > Change runtime type > **T4 GPU**. Needs `ball_colab.zip` in the top of your Drive (`python -m vision.ball_finetune --pack --roboflow data/roboflow-ball-v2` on the workstation).

Every session: run cells 1-3, then cell 5. Training resumes from `last.pt` on Drive, so a cut-off session loses at most the epoch it was in.

In [ ]:
# 1. Drive + GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code + the same ultralytics as the workstation (leave Colab's torch alone)
![ -d /content/repo ] || git clone -q https://github.com/juanmendoza-dev/ml-soccer-vision-model /content/repo
!git -C /content/repo pull -q
!pip -q install ultralytics==8.4.164

In [ ]:
# 3. Dataset onto the VM's own disk (Drive is too slow to train from), same path every session
![ -f /content/ball/dataset.yaml ] || (cp /content/drive/MyDrive/ball_colab.zip /content/ && unzip -q /content/ball_colab.zip -d /content && rm /content/ball_colab.zip)
!wc -l /content/ball/train.txt /content/ball/val.txt

## 4. Timing, first session only (F4 hard stop)

At most 2 min per batch size, writes nothing it keeps. An out-of-memory error on batch 5 is an answer (4 is the size), not a failure. Paste the `batch ...` lines back to Claude and wait for the go before cell 5.

In [ ]:
%cd /content/repo
!python -m vision.ball_finetune --timing --batch 4 5 --data /content/ball/dataset.yaml --weights /content/ball/football-ball-detection.pt

## 5. Train (after the go)

Runs go to `MyDrive/ball_runs/ball-ft/` (`weights/last.pt`, `weights/best.pt`, `results.csv`). Re-running this cell in a new session resumes.

In [ ]:
BATCH = 4  # set from the timing
%cd /content/repo
!python -m vision.ball_finetune --train --batch {BATCH} --data /content/ball/dataset.yaml --weights /content/ball/football-ball-detection.pt --project /content/drive/MyDrive/ball_runs